In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from dotenv import load_dotenv
import os
import yaml
import json
import pandas as pd

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from crewai import Agent, Task, Crew

from typing import List, Optional
from pydantic import BaseModel, Field

from crewai_tools import FileReadTool, ScrapeWebsiteTool, PDFSearchTool, SerperDevTool
from crewai.tools import BaseTool

/home/cloudcraftz/G/LLM/JobHunter-AI-Agent/venv/lib/python3.11/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/home/cloudcraftz/G/LLM/JobHunter-AI-Agent/venv/lib/python3.11/site-packages/crewai_tools/tools/scrapegraph_scrape_tool/scrapegraph_scrape_tool.py:34: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator("website_url")
/home/cloudcraftz/G/LLM/JobHunter-AI-Agent/venv/lib/python3.11/site-packages/crewai_tools/tools

In [3]:
_ = load_dotenv()
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
GOOGLE_API_KEY = os.environ['GOOGLE_API_KEY']
GROQ_API_KEY = os.environ['GROQ_API_KEY']
SERPER_API_KEY = os.environ['SERPER_API_KEY']

## LLM

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18")

## Output

In [5]:
class JobSearchedReport(BaseModel):
        
    job_title: str = Field(..., description="The title of the job position.")
    company: str = Field(..., description="The company offering the job.")
    location: str = Field(..., description="The location of the job.")
    relevance_score: int = Field(..., ge=0, le=100, description="Relevance score of the job (0-100%).")
    application_link: str = Field(..., description="The direct link to apply for the job.")
    job_description: Optional[str] = Field(None, description="Job description for relevant job positions.")
    salary_expectations: Optional[str] = Field(None, description="Expected salary range for relevant job positions.")


# class JobSearchedReport(BaseModel):
#     jobs: List[JobSearchedResponce] 

## Tools

In [6]:
import requests

class JobSearchTool(BaseTool):
    name: str = "Job Search Tool"
    description: str = "Search for job postings online using keywords and filters."

    def _run(self, keywords: str, location: str, experience: str, platform: str = "all", count: int = 5, days_ago: int = 7):
        if not SERPER_API_KEY:
            return "Serper API key not configured."

        try:
            url = "https://google.serper.dev/search"
            query = f"{keywords} jobs in {location} requiring {experience} years experience"
            if platform.lower() != "all":
                query += f" {platform}"

            params = {
                "engine": "google_jobs",
                "q": query,
                "api_key": SERPER_API_KEY,
                "hl": "en",
                "chips": f"date_posted:{days_ago}d"
            }

            response = requests.get(url, params=params)
            data = response.json()

            if "error" in data:
                return f"Serper API error: {data['error']}"

            jobs = []
            for job in data.get("jobs_results", [])[:count]:
                job_entry = {
                    "title": job.get("title", "Unknown Title"),
                    "company": job.get("company_name", "Unknown Company"),
                    "location": job.get("location", "Unknown Location"),
                    "description": job.get("description", "No description available"),
                    "application_link": job.get("apply_link", {}).get("link", "Not provided"),
                    "date_posted": job.get("detected_extensions", {}).get("posted_at", "Recent"),
                    "platform": job.get("via", "Unknown"),
                    "job_type": job.get("detected_extensions", {}).get("employment_type", "Not specified"),
                    "is_real_job": True
                }
                jobs.append(job_entry)

            return jobs

        except Exception as e:
            return f"Serper API search error: {str(e)}"


In [7]:
search_tool = SerperDevTool() #SearchTool()
scrape_tool = ScrapeWebsiteTool()
# read_resume = FileReadTool(file_path='/home/cloudcraftz/Downloads/Sumegh_Sen_Resume.pdf')
semantic_search_resume = PDFSearchTool(pdf='/home/cloudcraftz/Downloads/Sumegh_Sen_Resume.pdf')

## Define file paths for YAML configurations

In [8]:
# Define file paths for YAML configurations
files = {
    'agents': '/home/cloudcraftz/G/LLM/JobHunter-AI-Agent/src/job_hunter/config/agents.yaml',
    'tasks': '/home/cloudcraftz/G/LLM/JobHunter-AI-Agent/src/job_hunter/config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Creating Agents

In [9]:
# Creating Agents
job_hunter_agent = Agent(
  config=agents_config['Tech_Job_Hunter_Agent'],
  tools = [scrape_tool, search_tool],
  llm = llm,
  allow_delegation=False,
  verbose=True
)

job_filter_agent = Agent(
  config=agents_config['Tech_Job_Analyst_Agent'],
  tools = [semantic_search_resume],
  llm = llm,
  allow_delegation=True,
  verbose=True
)

report_writer_agent = Agent(
  config=agents_config['Report_Writer_Agent'],
  llm=llm,
  allow_delegation=True,
  verbose=True
)

## Creating Tasks

In [10]:
# Creating Tasks
research_task = Task(
  config=tasks_config['research_task'],
  tools = [scrape_tool, search_tool],  
  agent=job_hunter_agent
)

filter_task = Task(
  config=tasks_config['filter_task'],
  tools = [semantic_search_resume],
  agent=job_filter_agent
)

writer_task = Task(
  config=tasks_config['writer_task'],
  output_pydantic= JobSearchedReport,  
  context=[research_task, filter_task],
  agent=report_writer_agent,
)

## Creating Crew

In [11]:
# Creating Crew
job_hunter_crew = Crew(
    agents=[job_hunter_agent, job_filter_agent, report_writer_agent],
    tasks=[research_task, filter_task, writer_task],
    verbose=True
)


## Run

In [12]:
Skills = "Python, Machine Learning, Deep Learning, NLP, NumPy, Pandas, Generative AI, RAG, Agent AI, LangChain, CI/CD, Docker, TensorFlow, Keras, PyTorch, AWS"
Experience = "3-5 years"
Location = "India"
Keywords = "Data Scientist, AI/ML Engineer"
Timeframe = "7 days"
Relevance_threshold = "20"
number_of_job = "10"
platform_list = "Naukri.com, LinkedIn, Indeed, CutShort, Glassdoor, wellfound"

In [13]:
inputs = {
  "skills": Skills,
  "experience": Experience,
  "location": Location,
  "keywords": Keywords,
  "timeframe": Timeframe,
  "threshold": Relevance_threshold,
  "platforms": platform_list,
  "num_jobs": number_of_job
}

In [14]:
result = job_hunter_crew.kickoff(inputs=inputs)

 
[2025-03-22 10:02:46][🚀 CREW 'CREW' STARTED, B11FBE26-EF0B-4448-A607-7831A4D13A92]: 2025-03-22 10:02:46.645710
 
[2025-03-22 10:02:46][📋 TASK STARTED: SEARCH FOR JOB OPPORTUNITIES ON THE SPECIFIED PLATFORMS THAT MATCH A CANDIDATE'S PROFILE. ENSURE JOB POSTINGS ARE **VALID**, MEET THE GIVEN CRITERIA, AND WERE POSTED WITHIN THE REQUESTED TIMEFRAME. IF THE REQUIRED NUMBER OF JOBS IS NOT FOUND, RETRY THE SEARCH WITH VARIATIONS.
USE THE FOLLOWING CRITERIA TO SEARCH FOR JOBS:
  - SKILLS: PYTHON, MACHINE LEARNING, DEEP LEARNING, NLP, NUMPY, PANDAS, GENERATIVE AI, RAG, AGENT AI, LANGCHAIN, CI/CD, DOCKER, TENSORFLOW, KERAS, PYTORCH, AWS
  - YEARS OF EXPERIENCE: 3-5 YEARS
  - LOCATION: INDIA
  - KEYWORDS: DATA SCIENTIST, AI/ML ENGINEER
  - TIMEFRAME: JOBS POSTED IN THE PAST 7 DAYS DAYS (E.G., PAST 24 HOURS, PAST 7 DAYS, PAST 30 DAYS)
  - PLATFORMS: NAUKRI.COM, LINKEDIN, INDEED, CUTSHORT, GLASSDOOR, WELLFOUND
]: 2025-03-22 10:02:46.657219
 
[2025-03-22 10:02:46][🤖 AGENT 'TECH JOB HUNTER
' START

In [15]:
costs = 0.150 * (job_hunter_crew.usage_metrics.prompt_tokens + job_hunter_crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

Total costs: $0.0122


## Result

In [16]:
result_dict =  result.model_dump()

raw_text = result_dict['raw'] #result.model_dump_json() #

# Ensure it's a valid JSON array
wrapped_text = f"[{raw_text}]"

# Replace any incorrect trailing commas (if applicable)
wrapped_text = wrapped_text.replace("},\n{", "},{")

# Convert JSON text to Python list
job_listings = json.loads(wrapped_text)

# Convert to a Pandas DataFrame
df = pd.DataFrame(job_listings[0])

# Display the DataFrame
df

,job_title,company,location,relevance_score,application_link,job_description,salary_expectations
0,Engineer - Data Science,Pentair,Noida,100,https://www.naukri.com/job-listings-engineer-d...,Strong understanding of data science principle...,Not specified
1,AI / ML Engineer,Accenture,Bengaluru,90,https://www.naukri.com/job-listings-ai-ml-engi...,Developing AI applications and systems.,Not specified
2,Data Scientist,TIFIN AMP,"Boulder, CO",85,https://www.linkedin.com/jobs/data-scientist-jobs,Looking for a data scientist to analyze data a...,Not specified
3,AI/Machine Learning Engineer,Trovo Health,"New York, NY",80,https://www.linkedin.com/jobs/ai-machine-learn...,Building AI-based solutions in healthcare.,Not specified
4,Machine Learning Engineer,Ikigai,"San Mateo, CA",75,https://www.linkedin.com/jobs/machine-learning...,Develop and optimize machine learning models.,Not specified
5,Data Scientist with NLP/GenAI,Unknown,Bangalore,70,https://www.linkedin.com/posts/bhupender-singh...,Urgent requirement for a Data Scientist focusi...,Not specified
6,Data Scientist,The Walt Disney Studios,"Burbank, CA",65,https://www.linkedin.com/jobs/data-scientist-jobs,Analyze data to support creative decision-making.,Not specified
7,"Data Scientist II, Credit",Brex,"New York, NY",60,https://www.linkedin.com/jobs/data-scientist-jobs,Focus on credit and risk modeling.,Not specified
8,Machine Learning Engineer,"Tagup, Inc.","Boston, MA",55,https://www.linkedin.com/jobs/machine-learning...,Enhancing predictive maintenance solutions.,Not specified
9,Machine Learning Engineer - Early in Career,Splunk,"California, United States",50,https://www.linkedin.com/jobs/machine-learning...,Entry-level position in machine learning.,Not specified


In [17]:
df.to_csv("Saved_job_1.csv", index=False)

In [19]:
df['application_link'][5]

'https://www.linkedin.com/posts/bhupender-singh-6a220853_urgent-requirement-data-scientist-with-activity-7216715548590522368-AkMQ'